# Data Evaluation & Cleaning
Exploratory notebook for analysing dataset quality, running model evaluation on a held-out golden dataset, and detecting contamination between training and evaluation sets.

**Sections**
1. Model & golden dataset setup
2. Data quality analysis — exact duplicates, cross-label similarity, near-duplicates
3. Model evaluation on golden dataset — per-category accuracy, failure modes, Grad-CAM
4. Background removal with rembg (exploratory — ultimately abandoned due to domain mismatch)
5. Custom image inference
6. Watermark detection
7. Error analysis
8. Training dataset contamination check
9. Training set exact-duplicate removal

In [ ]:
import sys
import os
import cv2
from matplotlib import pyplot as plt
from safetensors.torch import load_file
import torch
import random
import torch.nn as nn
from torchvision.models import get_model, get_weight
from PIL import Image
from pathlib import Path
import pickle
sys.path.append("..")
sys.path.append(".")
from utils.generate_masks import generate_produce_mask
from utils.grade_produce import grade_colour, grade_colour_generic, grade_proportion
from utils.mtl_model import MultiTaskClassifier
from utils.dataset import ProduceDataset
from experiment_configs import task_2_config_final as experiments
import plotly.express as px


# Select experiment and trained weights path 
TRAINED_MODEL_PATH = Path(".") / "models" / "EX6a_SWIN_FINETUNE_MTL_20260422201021.safetensors"
EXPERIMENT = experiments.EX6a_SWIN_FINETUNE_MTL
assert EXPERIMENT.display_name.lower() in TRAINED_MODEL_PATH.stem.lower(), \
        (f"Checkpoint '{TRAINED_MODEL_PATH.name}"
         f"doesn't match experiment '{EXPERIMENT.display_name}"
         )
PRODUCE_DATESET_PATH = Path(".") / "data" / "Fruit_And_Vegetable_Diseases_Dataset"
# Type threshold given the limited number of produce types
# Inputs below the threshold will be considered "Uknown"
TYPE_CONFIDENCE_THRESHOLD = 50


# Load empty model architecture (without pretrained weights)
# Build the same MTL wrapper used during training
base_model = get_model(EXPERIMENT.architecture, weights=None)
produce_dataset = ProduceDataset(dataset_root_dir=PRODUCE_DATESET_PATH, 
                                 transform=None)

if EXPERIMENT.is_mtl:
    # MTL: instantiate custom MTL with auxillary produce type objective
     # Need num_types to match training - get from dataset
    model = MultiTaskClassifier(base_model, 
                                num_produce_classes=produce_dataset.num_produce_types)
else:
    # STL: replace the final classifier for binary health classification
    num_features = base_model.classifier[1].in_features
    base_model.classifier[1] = nn.Linear(num_features, 2)
    model = base_model

# Load trained weights into the MTL wrapper
if os.path.splitext(TRAINED_MODEL_PATH)[1] == ".pth":
    model.load_state_dict(torch.load(TRAINED_MODEL_PATH, weights_only=True))
else: # Load as safetensor
    state_dict = load_file(TRAINED_MODEL_PATH)
    model.load_state_dict(state_dict)

# Send to device and set model to eval mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Get transforms for inference input
pretrained_weights = get_weight(EXPERIMENT.weight_string)
auto_transforms = pretrained_weights.transforms()

# Define the target classes
health_names = ['Healthy', 'Rotten']
type_names = produce_dataset.produce_type_names

## 1. Load the Dataset
This cell loads the chosen dataset alongside the trained model to evaluate.

In [ ]:
import os
import sys
from pathlib import Path
from matplotlib import pyplot as plt
from PIL import Image

sys.path.append(".")
from utils.dataset import ProduceDataset
from torchvision.models import get_weight

# Reuse the same transforms the model was trained with
auto_transforms = pretrained_weights.transforms()

PRODUCE_DATASET_PATH = Path(".") / "data" / "Fruit_And_Vegetable_Diseases_Dataset_no_identical_no_aug" 
# Clean copy — for preprocessing and training
produce_dataset = ProduceDataset(dataset_root_dir=PRODUCE_DATASET_PATH, transform=auto_transforms)
print(f"Loaded {len(produce_dataset)} images")


In [ ]:
from pathlib import Path

DATASET_ROOT = Path(".") / "data" / "Fruit_And_Vegetable_Diseases_Dataset_no_identical"

for folder in sorted(DATASET_ROOT.iterdir()):
    if folder.is_dir():
        count = len(list(folder.glob("*.jpg"))) + len(list(folder.glob("*.png")))
        print(f"{folder.name:<30} {count}")


## 2. Data Quality Analysis
Check the dataset to surface duplicates and visually ambiguous images before using them for evaluation.

### 2a. Exact Duplicate Detection
Perceptual hashing (pHash) groups images with identical content. Keeping duplicates inflates per-category counts and biases evaluation metrics.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import imagehash
from collections import defaultdict
from pathlib import Path
from PIL import Image
from tqdm import tqdm

def hash_image(path):
    try:
        return path, str(imagehash.phash(Image.open(path)))
    except Exception as e:
        return path, None

hash_map = defaultdict(list)

for path in tqdm(produce_dataset.image_paths, desc="Hashing"):
    try:
        h = str(imagehash.phash(Image.open(path), hash_size=8))
        hash_map[h].append(path)
    except Exception as e:
        print(f"Could not open {path}: {e}")

duplicate_groups = {h: paths for h, paths in hash_map.items() if len(paths) > 1}
total_duplicates = sum(len(v) - 1 for v in duplicate_groups.values())

print(f"Unique images:    {len(hash_map) - len(duplicate_groups)}")
print(f"Duplicate groups: {len(duplicate_groups)}")
print(f"Redundant files:  {total_duplicates}")


if duplicate_groups:
    print("\nTop 10 largest duplicate groups:")
    for h, paths in sorted(duplicate_groups.items(), key=lambda x: -len(x[1]))[:10]:
        folders = set(Path(p).parent.name for p in paths)
        print(f"  {len(paths)} copies | folders: {folders}")
        for p in paths[:3]:
            print(f"    {Path(p).name}")
        if len(paths) > 3:
            print(f"    ... and {len(paths)-3} more")


In [ ]:
def show_duplicate_group(paths, max_shown=6):
    n = min(len(paths), max_shown)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, path in zip(axes, paths[:n]):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(f"{Path(path).parent.name}\n{Path(path).name}", fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Show the top 3 largest duplicate groups
for h, paths in sorted(duplicate_groups.items(), key=lambda x: -len(x[1]))[:3]:
    folders = set(Path(p).parent.name for p in paths)
    print(f"\n{len(paths)} copies — folders: {folders}")
    show_duplicate_group(paths)


### 2b. Near-Duplicate Detection
Scans within each produce folder for images that look nearly identical (pHash distance ≤ `THRESHOLD`). 

In [ ]:
import imagehash
from collections import defaultdict
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

THRESHOLD = 8  # hamming distance — tune this up/down

by_folder = defaultdict(list)
for path in produce_dataset.image_paths:
    by_folder[Path(path).parent.name].append(path)

near_dup_groups = []

for folder, paths in by_folder.items():
    hashes = []
    for path in paths:
        try:
            h = imagehash.phash(Image.open(path).convert('RGB'))
            hashes.append((path, h))
        except:
            pass

    used = set()
    for i, (path_a, hash_a) in enumerate(hashes):
        if path_a in used:
            continue
        group = [path_a]
        max_dist = 0
        for j, (path_b, hash_b) in enumerate(hashes):
            if i == j or path_b in used:
                continue
            dist = hash_a - hash_b
            if dist <= THRESHOLD:
                group.append(path_b)
                used.add(path_b)
                max_dist = max(max_dist, dist)
        if len(group) > 1:
            used.add(path_a)
            near_dup_groups.append((folder, group, max_dist))


print(f"Near-duplicate groups: {len(near_dup_groups)}")
print(f"Images involved: {sum(len(g) for _, g, _ in near_dup_groups)}")




In [ ]:

# Show bell pepper groups first
from itertools import islice

# Show up to 3 groups per folder, up to 6 images per group
current_folder = None
for folder, group, max_dist in near_dup_groups:
    if folder != current_folder:
        current_folder = folder
        folder_groups = [(f, g, d) for f, g, d in near_dup_groups if f == folder]
        print(f"\n{'='*50}")
        print(f"{folder} — {len(folder_groups)} groups")
        print(f"{'='*50}")
        shown = 0

    if shown >= 3:
        continue

    n = min(len(group), 6)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    if n == 1: axes = [axes]
    for ax, path in zip(axes, group[:n]):
        ax.imshow(Image.open(path))
        ax.set_title(Path(path).name[:20], fontsize=6)
        ax.axis('off')
    plt.suptitle(f"{folder} — {len(group)} near-dupes (dist={max_dist})", fontsize=9)
    plt.tight_layout()
    plt.show()
    shown += 1


In [ ]:
from collections import Counter

dist_counts = Counter(d for _, _, d in near_dup_groups)
for dist in sorted(dist_counts):
    cumulative = sum(dist_counts[d] for d in dist_counts if d <= dist)
    print(f"dist={dist}: {dist_counts[dist]} groups | cumulative ≤{dist}: {cumulative} groups")


### 2c. Cross-Label Contradictions
Checks whether any image hashes appear under both Healthy and Rotten labels. A single image labelled both ways would directly corrupt training signal.

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
from pathlib import Path

# Option B: ImageNet-only features (not your fine-tuned model)
# Swap in `model.backbone` if you want task-tuned features instead.
from torchvision.models import swin_s, Swin_S_Weights
feature_model = swin_s(weights=Swin_S_Weights.IMAGENET1K_V1)
feature_model.head = torch.nn.Identity()   # drop classifier, keep pooled features
feature_model = feature_model.to(device).eval()

# Dataset without augmentation, cleaned split
from utils.dataset import ProduceDataset
from torchvision.transforms import v2
NOIDENT = Path("data/Fruit_And_Vegetable_Diseases_Dataset_no_identical_no_aug")

transforms = Swin_S_Weights.IMAGENET1K_V1.transforms()
dataset = ProduceDataset(dataset_root_dir=NOIDENT, transform=transforms)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# Extract features
features, health_lbls, type_lbls, paths = [], [], [], []
with torch.no_grad():
    for imgs, y_h, y_t in tqdm(loader, desc="Embedding"):
        feats = feature_model(imgs.to(device))      # [B, 768]
        features.append(feats.cpu().numpy())
        health_lbls.extend(y_h.tolist())
        type_lbls.extend(y_t.tolist())
features = np.concatenate(features, axis=0)
health_lbls = np.array(health_lbls)
type_lbls = np.array(type_lbls)
paths = np.array(dataset.image_paths)

# Nearest neighbours (+1 because first match is self)
K = 10
nn = NearestNeighbors(n_neighbors=K + 1, metric="cosine").fit(features)
_, idx = nn.kneighbors(features)
neighbour_idx = idx[:, 1:]                           # drop self

# Agreement scores
health_agree = (health_lbls[neighbour_idx] == health_lbls[:, None]).mean(axis=1)
type_agree   = (type_lbls[neighbour_idx]   == type_lbls[:, None]).mean(axis=1)

# Flag candidates
AGREE_THRESHOLD = 0.3     # <30% neighbours share the label
health_suspicious = np.where(health_agree < AGREE_THRESHOLD)[0]
print(f"Suspicious health labels: {len(health_suspicious)} / {len(paths)} "
      f"({100 * len(health_suspicious) / len(paths):.2f}%)")

# Per-class rate — the interesting number for the report
for cls_idx, cls_name in enumerate(health_names):
    mask = health_lbls == cls_idx
    n_flagged = (health_agree[mask] < AGREE_THRESHOLD).sum()
    print(f"  {cls_name}: {n_flagged} / {mask.sum()} "
          f"({100 * n_flagged / mask.sum():.2f}%) flagged")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Sort by lowest agreement (most suspicious first)
ranked = health_suspicious[np.argsort(health_agree[health_suspicious])]
TO_SHOW = 24

fig, axes = plt.subplots(TO_SHOW, 6, figsize=(14, 2.5 * TO_SHOW))
for row, i in enumerate(ranked[:TO_SHOW]):
    # Image itself
    axes[row, 0].imshow(Image.open(paths[i]))
    axes[row, 0].set_title(
        f"FLAGGED: {health_names[health_lbls[i]]}\n"
        f"agree={health_agree[i]:.0%}",
        fontsize=8, color='red')
    axes[row, 0].axis('off')

    # 5 nearest neighbours
    for col, nb in enumerate(neighbour_idx[i, :5]):
        axes[row, col + 1].imshow(Image.open(paths[nb]))
        axes[row, col + 1].set_title(
            f"nn: {health_names[health_lbls[nb]]}", fontsize=7)
        axes[row, col + 1].axis('off')

plt.tight_layout()
plt.savefig("figures/nn_disagreement_gallery.png", dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd

# Build a dataframe from the existing arrays
flag_df = pd.DataFrame({
    "produce": [produce_dataset.produce_type_names[t] for t in type_lbls],
    "health":  [health_names[h] for h in health_lbls],
    "flagged": health_agree < AGREE_THRESHOLD,
})

# Pivot: rows = produce, cols = health, values = (flag_count, total, rate)
summary = (
    flag_df.groupby(["produce", "health"])
           .agg(flagged=("flagged", "sum"), total=("flagged", "size"))
           .assign(rate=lambda d: d["flagged"] / d["total"] * 100)
           .round(2)
           .reset_index()
)

# Pretty wide table
wide = summary.pivot(index="produce", columns="health",
                     values=["flagged", "total", "rate"])
print(wide)


In [ ]:
path_to_health = dict(zip(produce_dataset.image_paths, produce_dataset.health_lbls))

cross_label_groups = {
    h: paths for h, paths in duplicate_groups.items()
    if len({path_to_health[p] for p in paths}) > 1
}
print(f"Cross-label duplicate groups: {len(cross_label_groups)}")
print(f"Images with contradictory labels: {sum(len(v) for v in cross_label_groups.values())}")


## 3. Model Evaluation on Golden Dataset
Runs the trained checkpoint through the cleaned golden set and builds a results dataframe with per-image predictions, true labels, and loss. All subsequent analysis (accuracy, failure modes, Grad-CAM) is derived from this dataframe.

In [ ]:
from torch.utils.data import DataLoader
from utils.dataset import ProduceDataset
import pandas as pd
from matplotlib import pyplot as plt

PRODUCE_DATASET_PATH = Path(".") /".."/ "golden_dataset"

dataset = ProduceDataset(dataset_root_dir=PRODUCE_DATASET_PATH, transform=auto_transforms)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f"Loaded {len(dataset)} images")


In [ ]:
from tqdm import tqdm

criterion = torch.nn.CrossEntropyLoss(reduction='none')  # per-sample loss

all_losses, all_preds, all_labels, all_paths = [], [], [], []

with torch.no_grad():
    # ProduceDataset returns (image, y_health, y_type); MTL model returns (health_logits, type_logits)
    for i, (imgs, y_health, y_type) in enumerate(tqdm(loader, desc="Evaluating")):
        imgs, y_health = imgs.to(device), y_health.to(device)
        if EXPERIMENT.is_mtl:
            health_logits, _ = model(imgs)
        else:
            health_logits = model(imgs)
        losses = criterion(health_logits, y_health)
        preds = health_logits.argmax(dim=1)

        all_losses.extend(losses.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y_health.cpu().tolist())

    # Recover paths in the same order (DataLoader with shuffle=False preserves order)
    all_paths = dataset.image_paths

df = pd.DataFrame({
    "path": all_paths,
    "produce": [Path(p).parent.name.split("__")[0] for p in all_paths],
    "true_label": all_labels,
    "pred": all_preds,
    "correct": [p == l for p, l in zip(all_preds, all_labels)],
    "loss": all_losses,
})

print(df["correct"].mean(), "overall accuracy")

In [ ]:
per_produce = df.groupby("produce")["correct"].mean().sort_values()
print(per_produce.to_string())

per_produce.plot(kind="barh", figsize=(8, 5), color="steelblue")
plt.xlabel("Accuracy")
plt.title("Per-produce accuracy")
plt.axvline(x=df["correct"].mean(), color="red", linestyle="--", label="Overall avg")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#worst = df[df["correct"] == False].sort_values("loss", ascending=False).head(36)

best = df[df["correct"] == True].sort_values("loss", ascending=True).head(36)



fig, axes = plt.subplots(6, 6, figsize=(14, 14))
for ax, (_, row) in zip(axes.flat, best.iterrows()):
    ax.imshow(Image.open(row["path"]))
    true_name = health_names[row["true_label"]]
    pred_name = health_names[row["pred"]]

    ax.set_title(f"{row['produce']}\nTrue: {true_name} | Pred: {pred_name}\nLoss: {row['loss']:.2f}", fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
with torch.no_grad():
    feats = model.backbone.features(torch.randn(1, 3, 224, 224).to(device))
    print(feats.shape)   # expect [1, 7, 7, 768] for Swin-S


### 3a. Grad-CAM: Failure Mode Analysis
Grad-CAM highlights the image regions the model relied on when it predicted incorrectly. Common failure patterns: focusing on background colour (red soil → rotten), texture artifacts, or low-contrast rotten spots missed entirely.

In [ ]:
# pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np
from pytorch_grad_cam import GradCAMPlusPlus, EigenCAM, HiResCAM

class HealthHeadOnly(torch.nn.Module):
    def __init__(self, mtl_model):
        super().__init__()
        self.mtl = mtl_model
    def forward(self, x):
        health, _ = self.mtl(x)
        return health

cam_model = HealthHeadOnly(model).eval()
# Swin's last feature stage — verify with `print(model.backbone)` if unsure
target_layer = [model.backbone.features[-3]]
def reshape_transform_swin(tensor):
    # torchvision Swin features: [B, H, W, C] → [B, C, H, W]
    return tensor.permute(0, 3, 1, 2).contiguous()

from pytorch_grad_cam import GradCAMPlusPlus

cam = GradCAMPlusPlus(
    model=cam_model,
    target_layers=target_layer,
    reshape_transform=reshape_transform_swin,
)

#best = df[df["correct"]].sort_values("loss", ascending=True).head(32)
rotten_idx = health_names.index("Rotten")

import math

N = 120
PAIRS_PER_ROW = 1           # 3 pairs = 6 columns total
rows = (N + PAIRS_PER_ROW - 1) // PAIRS_PER_ROW

candidates = (
    df[(df["correct"]) & (df["true_label"] == rotten_idx)]
      .query("0.00 < loss < 3.0")
      .sort_values("loss", ascending=False)
      .head(N)
)

fig, axes = plt.subplots(rows, PAIRS_PER_ROW * 2,
                         figsize=(3 * PAIRS_PER_ROW * 2, 3.2 * rows))
axes = axes.reshape(rows, PAIRS_PER_ROW * 2)

for i, (_, row) in enumerate(candidates.iterrows()):
    r = i // PAIRS_PER_ROW
    c = (i % PAIRS_PER_ROW) * 2

    img = Image.open(row["path"]).convert('RGB')
    tensor = auto_transforms(img).unsqueeze(0).to(device)
    targets = [ClassifierOutputTarget(row["pred"])]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]

    img_resized = np.array(img.resize((224, 224))) / 255.0
    vis = show_cam_on_image(img_resized.astype(np.float32), grayscale_cam, use_rgb=True)

    confidence = math.exp(-row["loss"])   # p(correct class)

    # Left: original
    axes[r, c].imshow(img_resized)
    axes[r, c].set_title(f"{row['produce']} — {health_names[row['true_label']]}",
                         fontsize=8)
    axes[r, c].axis('off')

    # Right: CAM
    axes[r, c + 1].imshow(vis)
    axes[r, c + 1].set_title(f"pred {health_names[row['pred']]} · conf {confidence:.1%}",
                             fontsize=8)
    axes[r, c + 1].axis('off')

# Hide any unused axes
used = len(candidates)
for i in range(used, rows * PAIRS_PER_ROW):
    r = i // PAIRS_PER_ROW
    c = (i % PAIRS_PER_ROW) * 2
    axes[r, c].axis('off')
    axes[r, c + 1].axis('off')

plt.tight_layout()
#plt.savefig("figures/gradcam_paired.png", dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
print(f"df was built at... (add a timestamp in the eval cell)")
print(f"Model weights summary: {sum(p.sum().item() for p in model.parameters()):.4f}")


In [ ]:
print(f"rows in df: {len(df)}")
print(f"candidates: {len(candidates)}")
print(f"first candidate path: {candidates.iloc[0]['path']}")
print(f"first candidate conf: {math.exp(-candidates.iloc[0]['loss']):.1%}")


Class Imbalance


In [ ]:
from collections import Counter
from pathlib import Path
import pandas as pd

# Per-(produce, health) count from the cleaned dataset
counts = Counter()
for p in produce_dataset.image_paths:
    folder = Path(p).parent.name                # e.g. "Apple__Healthy"
    produce, health = folder.split("__")
    counts[(produce, health)] += 1

count_df = pd.DataFrame(
    [{"produce": k[0], "health": k[1], "count": v} for k, v in counts.items()]
)


In [ ]:
import plotly.express as px

# Order produce types by total count, biggest at top
order = count_df.groupby("produce")["count"].sum().sort_values().index.tolist()

fig = px.bar(
    count_df, y="produce", x="count", color="health",
    category_orders={"produce": order, "health": ["Healthy", "Rotten"]},
    color_discrete_map={"Healthy": "#10b981", "Rotten": "#d13333"},
    orientation='h',
    title="Class imbalance — some produce types have 10× more samples than others",
    text_auto=True
)
fig.update_layout(height=600, barmode='stack')
fig.show()


In [ ]:
fig = px.sunburst(
    count_df, path=["produce", "health"], values="count",
    color="health",
    color_discrete_map={"Healthy": "#10b981", "Rotten": "#ef4444"},
    title="Produce-type and health-label distribution",
)
fig.update_layout(height=700)
fig.show()


In [ ]:
pivot = count_df.pivot(index="produce", columns="health", values="count").fillna(0)
pivot["total"] = pivot.sum(axis=1)
pivot["ratio"] = pivot["Rotten"] / pivot["Healthy"]       # >1 = rotten-skewed
pivot = pivot.sort_values("ratio")

fig = px.bar(
    pivot.reset_index(), y="produce", x="ratio",
    orientation='h',
    title="Rotten / Healthy ratio per produce — 1.0 is balanced",
    labels={"ratio": "Rotten ÷ Healthy"},
    color="ratio",
    color_continuous_scale="RdYlGn_r",
    color_continuous_midpoint=1.0,
)
fig.add_vline(x=1.0, line_dash="dash", line_color="black",
              annotation_text="balanced", annotation_position="top right")
fig.update_layout(height=600, showlegend=False)
fig.show()


## 4. Background Removal (rembg) — Exploratory
Attempted U2Net-based background removal to reduce background bias observed in Grad-CAM. Grey background was substituted in place of the original. **Abandoned** — evaluation on the golden set dropped to ~49% (near random), likely because the segmentation model struggles with rotten produce and the grey-background domain is never seen during training.

In [ ]:
from rembg import remove, new_session
from PIL import Image
from pathlib import Path
from tqdm import tqdm

DATASET_ROOT = Path("data/Fruit_And_Vegetable_Diseases_Dataset_no_dups_no_aug_no_bg")
TARGET_CATEGORIES = [
    "Apple", "Banana", "Bellpepper", "Carrot", "Cucumber",
    "Grape", "Guava", "Jujube", "Mango", "Orange",
    "Pomegranate", "Potato", "Strawberry", "Tomato"
]

GREY = (128, 128, 128)
session = new_session("u2net", providers=["CUDAExecutionProvider"])

for category in TARGET_CATEGORIES:
    for split in ["Healthy", "Rotten"]:
        folder = DATASET_ROOT / f"{category}__{split}"
        if not folder.exists():
            continue
        images = list(folder.glob("*.jpg")) + list(folder.glob("*.png"))
        print(f"Processing {category}__{split}: {len(images)} images")
        for img_path in tqdm(images):
            try:
                img = Image.open(img_path).convert("RGB")
                result = remove(img, session=session)
                bg = Image.new("RGB", result.size, GREY)
                bg.paste(result, mask=result.split()[3])
                bg.save(img_path)
            except Exception as e:
                print(f"Failed: {img_path.name} — {e}")

print("Done")


## 5. Custom Image Inference
Quick sanity-check: run the model on a folder of arbitrary images and display predictions with confidence scores. Useful for testing the model on produce photos not from either dataset.

In [ ]:
# ── Quick inference on unseen images ──────────────────────────────────────────
from PIL import Image
import os

CUSTOM_IMAGES_DIR = Path("C:/Users/kxklm/AAI/AAI/Grape")  # put your images here
# folder structure doesn't matter — just point at the folder with your images

results = []
image_paths = list(CUSTOM_IMAGES_DIR.glob("*.jpg")) + list(CUSTOM_IMAGES_DIR.glob("*.png")) + list(CUSTOM_IMAGES_DIR.glob("*.jpeg"))

for img_path in image_paths:
    img = Image.open(img_path).convert("RGB")
    tensor = auto_transforms(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1)[0]
        pred = output.argmax(dim=1).item()
    
    results.append({
        "file": img_path.name,
        "pred": class_names[pred],
        "confidence": probs[pred].item(),
        "healthy_prob": probs[0].item(),   # adjust index if your class order differs
        "rotten_prob": probs[1].item(),
        "pred_idx": pred,
    })

# Display as grid with predictions
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for ax, r, img_path in zip(axes.flat, results, image_paths):
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    color = "green" if r["pred"] == "Healthy" else "red"
    ax.set_title(f"{r['pred']}\n{r['confidence']:.1%}", color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

pd.DataFrame(results)


In [ ]:
# ── Grad-CAM on golden test images ────────────────────────────────────────────
target_layer = [model.features[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

for r, img_path in zip(results, image_paths):
    img = Image.open(img_path).convert("RGB")
    tensor = auto_transforms(img).unsqueeze(0).to(device)

    targets = [ClassifierOutputTarget(r["pred_idx"])]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]

    img_resized = np.array(img.resize((384, 384))) / 255.0
    visualisation = show_cam_on_image(img_resized.astype(np.float32), grayscale_cam, use_rgb=True)

    plt.imshow(visualisation)
    plt.title(f"Pred: {r['pred']}  ({r['confidence']:.1%})", fontsize=9)
    plt.axis("off")
    plt.show()


## 6. Watermark Detection
Uses Tesseract OCR to scan every image for agency watermarks (Getty, Shutterstock, Alamy …). Watermarked images could be a source of spurious texture features — the model may learn to associate the watermark overlay with a class label rather than the produce itself.

In [ ]:
# pip install pytesseract pillow
# Also needs: https://github.com/UB-Mannheim/tesseract/wiki (Windows installer)
import pytesseract
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# Point to your tesseract install after installing
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

WATERMARK_KEYWORDS = ['getty', 'alamy', 'shutterstock', 'dreamstime', 'istock', '123rf', 'depositphotos']

watermarked = defaultdict(list)

for path in tqdm(clean_dataset.image_paths):
    try:
        text = pytesseract.image_to_string(Image.open(path)).lower()
        for keyword in WATERMARK_KEYWORDS:
            if keyword in text:
                watermarked[keyword].append(path)
                break
    except:
        pass

print(f"\nWatermarked images found: {sum(len(v) for v in watermarked.values())}")
for keyword, paths in sorted(watermarked.items(), key=lambda x: -len(x[1])):
    # Break down by folder
    folders = defaultdict(int)
    for p in paths:
        folders[Path(p).parent.name] += 1
    print(f"\n  {keyword}: {len(paths)} images")
    for folder, count in sorted(folders.items(), key=lambda x: -x[1]):
        print(f"    {folder:<30} {count}")


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

for agency, paths in sorted(watermarked.items()):
    print(f"\n{'='*60}")
    print(f"{agency.upper()} — {len(paths)} images")
    print(f"{'='*60}")
    
    cols = 5
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
    axes = axes.flat if rows > 1 else iter(axes)
    
    for ax, path in zip(axes, paths):
        ax.imshow(Image.open(path))
        ax.set_title(f"{Path(path).parent.name}\n{Path(path).name[:20]}", fontsize=6)
        ax.axis('off')
    
    # hide unused axes
    for ax in axes:
        ax.axis('off')
    
    plt.suptitle(f"{agency} watermarks", fontsize=10)
    plt.tight_layout()
    plt.show()


## 7. Error Analysis by Category
Breaks down model errors per produce type — showing the hardest failures and whether errors tend toward false-positive (Rotten predicted as Healthy) or false-negative (Healthy predicted as Rotten). Categories with visually similar Healthy/Rotten appearance (Strawberry, Pomegranate) are expected to dominate.

In [ ]:
N_WORST = 6  # images to show per produce

for produce in ["Grape", "Pomegranate", "Jujube", "Potato"]:
    subset = df[(df["produce"] == produce) & (df["correct"] == False)].sort_values("loss", ascending=False).head(N_WORST)
    
    if subset.empty:
        print(f"{produce}: no errors found")
        continue

    fig, axes = plt.subplots(1, len(subset), figsize=(3 * len(subset), 3))
    if len(subset) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, subset.iterrows()):
        ax.imshow(Image.open(row["path"]))
        ax.set_title(f"True: {class_names[row['true_label']]}\nPred: {class_names[row['pred']]}\nLoss: {row['loss']:.2f}", fontsize=7)
        ax.axis('off')
    plt.suptitle(f"{produce} — worst errors", fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
errors = df[df["correct"] == False].copy()
errors["error_type"] = errors.apply(
    lambda r: "Healthy → Rotten" if r["true_label"] == 0 else "Rotten → Healthy",
    axis=1
)

# Count errors per produce+type, divide by total images in that produce
total_per_produce = df.groupby("produce").size()
breakdown = errors.groupby(["produce", "error_type"]).size().unstack(fill_value=0)
breakdown_pct = (breakdown.div(total_per_produce, axis=0) * 100).round(1)

print(breakdown_pct.to_string())

breakdown_pct.plot(kind="barh", figsize=(9, 6), color=["tomato", "steelblue"])
plt.title("Error rate per produce (% of total images in that category)")
plt.xlabel("Error rate (%)")
plt.tight_layout()
plt.show()



## 8. Training Dataset Contamination Check
Hashes every image in the dirty training set and checks whether any match images in the golden evaluation set. Contamination means the model has seen evaluation images during training, inflating golden-set accuracy. Any matches found are removed from the golden set.

In [ ]:
import imagehash
from PIL import Image
from pathlib import Path
from tqdm import tqdm

GOLDEN_PATH = Path("../golden_dataset")
DIRTY_PATH = Path("data/Fruit_And_Vegetable_Diseases_Dataset")

print("Hashing golden dataset...")
golden_hashes = {}
for img_path in tqdm(list(GOLDEN_PATH.rglob("*.jpg")) + list(GOLDEN_PATH.rglob("*.png")) + list(GOLDEN_PATH.rglob("*.jpeg"))):
    try:
        h = str(imagehash.phash(Image.open(img_path).convert("RGB")))
        golden_hashes[h] = img_path
    except:
        pass

print(f"Golden images hashed: {len(golden_hashes)}")

print("\nHashing dirty dataset...")
contaminated = []
for img_path in tqdm(list(DIRTY_PATH.rglob("*.jpg")) + list(DIRTY_PATH.rglob("*.png"))):
    try:
        h = str(imagehash.phash(Image.open(img_path).convert("RGB")))
        if h in golden_hashes:
            contaminated.append((img_path, golden_hashes[h]))
    except:
        pass

print(f"\nContaminated images found: {len(contaminated)}")
for dirty_path, golden_path in contaminated[:20]:
    print(f"  DIRTY:  {dirty_path.parent.name}/{dirty_path.name}")
    print(f"  GOLDEN: {golden_path.parent.name}/{golden_path.name}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(contaminated), 2, figsize=(6, 3 * len(contaminated)))

for i, (dirty_path, golden_path) in enumerate(contaminated):
    axes[i, 0].imshow(Image.open(dirty_path))
    axes[i, 0].set_title(f"DIRTY\n{dirty_path.parent.name}\n{dirty_path.name[:25]}", fontsize=7)
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(Image.open(golden_path))
    axes[i, 1].set_title(f"GOLDEN\n{golden_path.parent.name}\n{golden_path.name[:25]}", fontsize=7)
    axes[i, 1].axis("off")

plt.suptitle(f"{len(contaminated)} contaminated image pairs", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Remove contaminated images from golden dataset
for _, golden_path in contaminated:
    try:
        golden_path.unlink()
        print(f"Deleted: {golden_path.name}")
    except Exception as e:
        print(f"Failed: {golden_path.name} — {e}")

print(f"\nGolden dataset cleaned — {len(contaminated)} images removed")


## 9. Training Set Exact-Duplicate Removal
Removes exact duplicate images from the training dataset using pHash. Keeping duplicates means the model sees the same image multiple times per epoch, which wastes capacity and biases it toward over-represented images. Hash results are cached to avoid rehashing on subsequent runs.

In [ ]:
import pickle
from pathlib import Path

HASH_CACHE = Path("data/dirty_hashes.pkl")

if HASH_CACHE.exists():
    print("Loading cached hashes...")
    with open(HASH_CACHE, "rb") as f:
        dirty_hashes = pickle.load(f)
    print(f"Loaded {len(dirty_hashes)} hashes")
else:
    print("Hashing dirty dataset...")
    dirty_hashes = {}
    for img_path in tqdm(list(DIRTY_PATH.rglob("*.jpg")) + list(DIRTY_PATH.rglob("*.png"))):
        try:
            h = str(imagehash.phash(Image.open(img_path).convert("RGB")))
            dirty_hashes[h] = img_path
        except:
            pass
    with open(HASH_CACHE, "wb") as f:
        pickle.dump(dirty_hashes, f)
    print(f"Hashed and cached {len(dirty_hashes)} images")


### 9a. Identify and Remove Exact Duplicates
Builds a hash map across the full training set, groups images with identical hashes, and deletes all but one representative per group. Result: 29,197 → 25,816 unique images (3,381 exact duplicates removed).

In [ ]:
import pickle
from pathlib import Path
from PIL import Image
import imagehash
from tqdm import tqdm
from collections import defaultdict
import os

DIRTY_PATH = Path("data/Fruit_And_Vegetable_Diseases_Dataset_no_identical")

# Build full map: hash → list of all paths
print("Hashing dirty dataset (tracking all duplicates)...")
full_hash_map = defaultdict(list)
for img_path in tqdm(list(DIRTY_PATH.rglob("*.jpg")) + list(DIRTY_PATH.rglob("*.png"))):
    try:
        h = str(imagehash.phash(Image.open(img_path).convert("RGB")))
        full_hash_map[h].append(img_path)
    except:
        pass

exact_dup_groups = {h: paths for h, paths in full_hash_map.items() if len(paths) > 1}
to_delete = [p for paths in exact_dup_groups.values() for p in paths[1:]]

print(f"Unique images: {len(full_hash_map)}")
print(f"Exact duplicate groups: {len(exact_dup_groups)}")
print(f"Files to delete: {len(to_delete)}")


In [ ]:
for path in to_delete:
    path.unlink()
print("Done")
